# 01 数据采集结果验证与分析

**目的**：验证并分析 NB00 (`00_tabular_data_download.ipynb`) 产生的数据采集产物。

**产物由 NB00 生成，本 notebook 仅做加载、验证和分析。**

**输入（NB00 产物）**：
- `tmp/content/candidates.parquet` — 筛选后的候选数据集池
- `tmp/content/slug_to_ref.csv` — Kaggle API slug→ref 映射
- `tmp/content/d_content.parquet` — 最终 D_content 子集
- `tmp/content/main_tables.parquet` — 已选主表文件注册表
- `data/tabular_raw/{Id}/` — 已下载的数据集文件

In [ ]:
# ---- 配置与导入 ----
import os, sys, json, re, csv
from pathlib import Path

import numpy as np
import pandas as pd

csv.field_size_limit(sys.maxsize)

# 路径配置
ROOT        = Path(".").resolve().parent.parent
DATA_DIR    = ROOT / "data"
RAW_DIR     = DATA_DIR / "raw_data"
TMP_DIR     = ROOT / "tmp"
CONTENT_DIR = TMP_DIR / "content"
TAB_RAW_DIR = DATA_DIR / "tabular_raw"

sys.path.insert(0, str(ROOT))
from src.content.sampling import read_by_ext

# 预算参数（与 NB00 保持一致）
B_DS       = 1000
MIN_VIEWS  = 1000
SIZE_MIN   = 100 * 1024
SIZE_MAX   = 100 * 1024 * 1024

TABULAR_TAGS = {
    "tabular", "classification", "regression",
    "exploratory data analysis", "data analytics",
}

print(f"ROOT: {ROOT}")
print(f"CONTENT_DIR: {CONTENT_DIR}")

In [ ]:
# ---- 加载元数据 ----
meta = pd.read_csv(DATA_DIR / "metadata_merged.csv", engine="python")
print(f"metadata_merged: {meta.shape}")

dv_cols = ["Id", "DatasetId", "VersionNumber", "TotalCompressedBytes"]
dv = pd.read_csv(RAW_DIR / "DatasetVersions.csv", usecols=dv_cols, engine="python")
print(f"DatasetVersions: {dv.shape}")

# Drop rows with missing DatasetId or VersionNumber before grouping
dv = dv.dropna(subset=["DatasetId", "VersionNumber"])
dv["DatasetId"] = dv["DatasetId"].astype(int)

idx_latest = dv.groupby("DatasetId")["VersionNumber"].idxmax()
idx_latest = idx_latest.dropna()
dv_latest = dv.loc[idx_latest, ["DatasetId", "TotalCompressedBytes"]].copy()
dv_latest.rename(columns={"DatasetId": "Id"}, inplace=True)

meta = meta.merge(dv_latest, on="Id", how="left")
print(f"After join: {meta.shape}, TotalCompressedBytes notna: {meta['TotalCompressedBytes'].notna().sum()}")
del dv

In [ ]:
# ---- 筛选候选集（交互式分析） ----
def has_tabular_tag(tags_str):
    if pd.isna(tags_str) or not isinstance(tags_str, str):
        return False
    tags = {t.strip().lower() for t in tags_str.split(",")}
    return bool(tags & TABULAR_TAGS)


def filter_candidates(df, min_views=MIN_VIEWS, size_min=SIZE_MIN, size_max=SIZE_MAX):
    mask_tag = df["Tags"].apply(has_tabular_tag)
    mask_size = (
        df["TotalCompressedBytes"].notna()
        & (df["TotalCompressedBytes"] >= size_min)
        & (df["TotalCompressedBytes"] <= size_max)
    )
    mask_views = df["TotalViews"] >= min_views

    print(f"Tag filter:   {mask_tag.sum():>7,} datasets")
    print(f"Size filter:  {mask_size.sum():>7,} datasets")
    print(f"Views filter: {mask_views.sum():>7,} datasets")

    combined = mask_tag & mask_size & mask_views
    print(f"Combined:     {combined.sum():>7,} datasets")
    return df[combined].copy()


candidates = filter_candidates(meta)

print("\nPool sizes for different B_ds budgets:")
for b in [500, 1000, 2000]:
    pool = candidates.nlargest(b, "TotalDownloads")
    print(f"  B_ds={b:>5d}: {len(pool):>5d} datasets, "
          f"min views={pool['TotalViews'].min():,.0f}, "
          f"min downloads={pool['TotalDownloads'].min():,.0f}")

In [ ]:
# ---- 加载 d_content ----
dc_path = CONTENT_DIR / "d_content.parquet"
assert dc_path.exists(), f"d_content.parquet not found at {dc_path}. Run NB00 first."

d_content = pd.read_parquet(dc_path, engine="fastparquet")
print(f"d_content: {d_content.shape}")
print(f"  Columns: {list(d_content.columns)}")
print(f"  doc_idx range: [{d_content['doc_idx'].min()}, {d_content['doc_idx'].max()}]")
print(f"  TotalDownloads range: [{d_content['TotalDownloads'].min():,.0f}, {d_content['TotalDownloads'].max():,.0f}]")

# ref 分布
has_ref = (d_content["ref"] != "").sum()
print(f"\n  Has ref: {has_ref}/{len(d_content)} ({has_ref/len(d_content)*100:.1f}%)")
d_content.head()

In [ ]:
# ---- API 匹配质量分析 ----
slug_ref_path = CONTENT_DIR / "slug_to_ref.csv"
assert slug_ref_path.exists(), f"slug_to_ref.csv not found. Run NB00 first."

slug_to_ref = pd.read_csv(slug_ref_path)
print(f"slug_to_ref: {len(slug_to_ref)} rows")
print(f"\nConfidence breakdown:")
print(slug_to_ref["confidence"].value_counts().to_string())

# 抽样 10 条 fuzzy 匹配供人工检查
fuzzy = slug_to_ref[slug_to_ref["confidence"] == "fuzzy"]
if len(fuzzy) > 0:
    sample_n = min(10, len(fuzzy))
    sample = fuzzy.sample(sample_n, random_state=42)
    print(f"\nFuzzy match samples ({sample_n} rows) for manual inspection:")
    for _, r in sample.iterrows():
        print(f"  Slug={r['Slug']}, Title={r['Title']}, ref={r['ref']}")
else:
    print("\nNo fuzzy matches found.")

In [ ]:
# ---- 下载覆盖分析 ----
dc_ids = set(d_content["Id"].astype(int).values)

existing_dirs = set()
empty_dirs = set()
if TAB_RAW_DIR.exists():
    for d in TAB_RAW_DIR.iterdir():
        if d.is_dir():
            try:
                did = int(d.name)
            except ValueError:
                continue
            if any(d.iterdir()):
                existing_dirs.add(did)
            else:
                empty_dirs.add(did)

downloaded = dc_ids & existing_dirs
empty = dc_ids & empty_dirs
missing = dc_ids - existing_dirs - empty_dirs

print(f"d_content IDs: {len(dc_ids)}")
print(f"  Downloaded (non-empty dir): {len(downloaded)} ({len(downloaded)/len(dc_ids)*100:.1f}%)")
print(f"  Empty dirs:                 {len(empty)}")
print(f"  Missing dirs:               {len(missing)}")

# 额外目录（不在 d_content 中但在磁盘上）
extra = (existing_dirs | empty_dirs) - dc_ids
if extra:
    print(f"  Extra dirs (not in d_content): {len(extra)}")

In [ ]:
# ---- 加载 main_tables ----
mt_path = CONTENT_DIR / "main_tables.parquet"
assert mt_path.exists(), f"main_tables.parquet not found. Run NB00 first."

main_tables = pd.read_parquet(mt_path, engine="fastparquet")
print(f"main_tables: {main_tables.shape}")

n_found = (main_tables["main_table_path"] != "").sum()
n_total = len(main_tables)
print(f"  Tables found: {n_found}/{n_total} ({n_found/n_total*100:.1f}%)")

if n_found > 0:
    found = main_tables[main_tables["main_table_path"] != ""]
    print(f"\n  Extension breakdown:")
    print(found["extension"].value_counts().to_string(header=False))
    print(f"\n  File size stats:")
    print(f"    Median: {found['file_size'].median()/1024:.1f} KB")
    print(f"    Mean:   {found['file_size'].mean()/1024:.1f} KB")
    print(f"    Max:    {found['file_size'].max()/1024/1024:.1f} MB")
    print(f"    Min:    {found['file_size'].min()/1024:.1f} KB")

In [ ]:
# ---- 数据质量抽样 ----
# 随机选 5 个数据集，用 read_by_ext() 读前几行验证可读性
found = main_tables[main_tables["main_table_path"] != ""].copy()
if len(found) >= 5:
    sample = found.sample(5, random_state=42)
elif len(found) > 0:
    sample = found
else:
    sample = pd.DataFrame()
    print("No main tables found to sample.")

for _, row in sample.iterrows():
    path = row["main_table_path"]
    ds_id = row["DatasetId"]
    print(f"\n--- DatasetId={ds_id}, ext={row['extension']}, size={row['file_size']/1024:.1f}KB ---")
    print(f"    Path: {path}")
    df = read_by_ext(path, nrows=5)
    if df is not None:
        print(f"    Shape: {df.shape}")
        print(f"    Columns: {list(df.columns)[:10]}{'...' if len(df.columns) > 10 else ''}")
        print(df.head(3).to_string())
    else:
        print(f"    FAILED to read")

In [ ]:
# ---- 管线 Yield 漏斗图 ----
n_candidates = len(candidates)
n_matched = (slug_to_ref["confidence"] != "none").sum() if len(slug_to_ref) > 0 else 0
n_dcontent = len(d_content)
n_downloaded_ct = len(downloaded)
n_has_table = n_found

stages = [
    ("候选池 (tag+size+views)", n_candidates),
    ("API 匹配 (exact+fuzzy)", n_matched),
    (f"d_content (top {B_DS})", n_dcontent),
    ("已下载 (非空目录)", n_downloaded_ct),
    ("有主表", n_has_table),
]

print("Pipeline Yield Funnel")
print("=" * 50)
max_bar = 40
max_count = max(s[1] for s in stages) if stages else 1
for label, count in stages:
    bar_len = int(count / max_count * max_bar) if max_count > 0 else 0
    bar = "\u2588" * bar_len
    pct = count / stages[0][1] * 100 if stages[0][1] > 0 else 0
    print(f"  {label:<28s} {count:>6,}  ({pct:>5.1f}%)  {bar}")
print("\n分析完成。")

## 总结

本 notebook 对 NB00 的采集产物进行了全面验证：
1. **候选集筛选** — tag/size/views 三重过滤结果分析
2. **API 匹配质量** — confidence 分布 + fuzzy 抽样检查
3. **下载覆盖** — d_content vs tabular_raw 目录对比
4. **主表覆盖** — 覆盖率 + 扩展名/文件大小分布
5. **数据可读性** — 随机抽样验证
6. **漏斗图** — 端到端 yield 可视化

**下一步**：运行 `02_content_view_construction.ipynb` 进行表格剖析和内容嵌入构建。